# Notebook goal
```text
Input:

ride_lakehouse_dev.silver.drivers
ride_lakehouse_dev.silver.rides
ride_lakehouse_dev.silver.payments

Output:

ride_lakehouse_dev.gold.driver_stats
ride_lakehouse_dev.gold.daily_stats

Gold responsibilities:

- join related Silver entities
- calculate business metrics
- aggregate detailed records
- provide simple tables for analytics / BI users
```

In [0]:
from pyspark.sql import functions as F

# Read data from Silver
In this notebook, we don't use stram because our data is small and calculating gold layer aggregations would be instant. But If our data grows too much we can use: change Data Feed, materialized views, or streaming.

In [0]:
silver_drivers = spark.table(
    "ride_lakehouse_dev.silver.drivers"
)

silver_rides = spark.table(
    "ride_lakehouse_dev.silver.rides"
)

silver_payments = spark.table(
    "ride_lakehouse_dev.silver.payments"
)

# Driver Stats Table
In this table I want to display each driver's performance like total rides, renevue, avarage ride distance. For this I need to join all tables.

In [0]:
ride_payment_df = (
    silver_rides.alias("r")

    .join(
        silver_payments.alias("p"),
        F.col("r.ride_id") == F.col("p.ride_id"),
        "left"
    )
)

In [0]:
# display(
#     ride_payment_df.select(
#         F.col("r.ride_id"),
#         F.col("r.driver_id"),
#         F.col("r.distance_km"),
#         F.col("r.fare_amount"),
#         F.col("p.amount").alias("payment_amount"),
#         F.col("r.ride_timestamp")
#     )
# )

ride_id,driver_id,distance_km,fare_amount,payment_amount,ride_timestamp
R-20260929114916836637-0017,D-20260929114916836637-000,11.23,24.45,24.45,2026-09-29T11:49:16.841Z
R-20260929114544667600-0009,D-20260929114544667600-001,7.65,24.23,24.23,2026-09-29T11:45:44.670Z
R-20260929114913448298-0016,D-20260929114913448298-006,7.17,25.86,25.86,2026-09-29T11:49:13.452Z
R-20260929114909008713-0030,D-20260929114909008713-009,14.62,43.13,43.13,2026-09-29T11:49:09.015Z
R-20260929114913448298-0028,D-20260929114913448298-000,8.14,22.58,22.58,2026-09-29T11:49:13.454Z
R-20260929114544667600-0036,D-20260929114544667600-008,6.18,19.42,19.42,2026-09-29T11:45:44.675Z
R-20260929114916836637-0031,D-20260929114916836637-001,13.57,28.15,28.15,2026-09-29T11:49:16.844Z
R-20260929114544667600-0023,D-20260929114544667600-000,16.86,48.45,48.45,2026-09-29T11:45:44.672Z
R-20260929114544667600-0005,D-20260929114544667600-000,14.06,43.08,43.08,2026-09-29T11:45:44.670Z
R-20260929114916836637-0027,D-20260929114916836637-002,11.14,37.98,37.98,2026-09-29T11:49:16.843Z


In [0]:
driver_ride_metrics = (
    ride_payment_df
    .groupBy( F.col("r.driver_id").alias("driver_id"))
    .agg(
        F.countDistinct("r.ride_id").alias("total_rides"),
        F.round(F.sum("r.distance_km"),2).alias("total_distance_km"),
        F.round(F.avg("r.distance_km"),2).alias("average_ride_distance_km"),
        F.round(
            F.sum(
                F.coalesce( # replace null values with 0.0
                    F.col("p.amount"),
                    F.lit(0.0)
                )
            ),
            2
        ).alias("total_revenue"),

        F.round(F.avg("p.amount"),2).alias("average_ride_revenue"),
        F.max("r.ride_timestamp").alias("last_ride_timestamp")
    )
)

In [0]:
driver_stats = (
    silver_drivers.alias("d")

    .join(
        driver_ride_metrics.alias("m"),
        F.col("d.id") == F.col("m.driver_id"),
        "left"
    )

    .select(
        F.col("d.id").alias("driver_id"),
        F.col("d.name").alias("driver_name"),
        F.col("d.car_number"),
        F.col("d.rating"),
        F.col("d.experience"),

        F.coalesce(
            F.col("m.total_rides"),
            F.lit(0)
        ).alias("total_rides"),

        F.coalesce(
            F.col("m.total_distance_km"),
            F.lit(0.0)
        ).alias("total_distance_km"),

        F.coalesce(
            F.col("m.average_ride_distance_km"),
            F.lit(0.0)
        ).alias("average_ride_distance_km"),

        F.coalesce(
            F.col("m.total_revenue"),
            F.lit(0.0)
        ).alias("total_revenue"),

        F.coalesce(
            F.col("m.average_ride_revenue"),
            F.lit(0.0)
        ).alias("average_ride_revenue"),

        F.col("m.last_ride_timestamp"),

        F.current_timestamp().alias(
            "_gold_updated_at"
        )
    )
)

In [0]:
# display(
#     driver_stats.orderBy(
#         F.col("total_revenue").desc()
#     )
# )

driver_id,driver_name,car_number,rating,experience,total_rides,total_distance_km,average_ride_distance_km,total_revenue,average_ride_revenue,last_ride_timestamp,_gold_updated_at
D-20260929114913448298-006,Mary Hunt,FR-0907,3.61,4,9,143.23,15.91,361.46,40.16,2026-09-29T11:49:13.456Z,2026-09-30T08:51:07.194Z
D-20260929114544667600-004,Harold Williams,ZD-5461,3.86,13,8,125.35,15.67,345.5,43.19,2026-09-29T11:45:44.677Z,2026-09-30T08:51:07.194Z
D-20260929114909008713-009,Karen Morgan,YK-0333,1.07,18,9,133.46,14.83,344.99,38.33,2026-09-29T11:49:09.020Z,2026-09-30T08:51:07.194Z
D-20260929114909008713-007,Joseph Moran,MN-5529,2.22,3,7,128.83,18.4,327.52,46.79,2026-09-29T11:49:09.018Z,2026-09-30T08:51:07.194Z
D-20260929114913448298-009,Elizabeth Hernandez,RL-9618,2.25,16,7,111.58,15.94,315.2,45.03,2026-09-29T11:49:13.457Z,2026-09-30T08:51:07.194Z
D-20260929114913448298-005,Virginia Schneider,SS-6638,1.33,17,7,102.5,14.64,283.17,40.45,2026-09-29T11:49:13.456Z,2026-09-30T08:51:07.194Z
D-20260929114544667600-008,Albert Perez,OW-0035,4.26,9,7,85.67,12.24,263.68,37.67,2026-09-29T11:45:44.676Z,2026-09-30T08:51:07.194Z
D-20260929114916836637-001,Vanessa Ray,DF-7414,2.63,18,7,101.15,14.45,255.34,36.48,2026-09-29T11:49:16.847Z,2026-09-30T08:51:07.194Z
D-20260929114909008713-001,Joe Wang,PC-8279,4.28,4,5,93.81,18.76,252.95,50.59,2026-09-29T11:49:09.015Z,2026-09-30T08:51:07.194Z
D-20260929114916836637-005,Lisa Young,TI-3588,4.87,6,5,96.29,19.26,251.5,50.3,2026-09-29T11:49:16.844Z,2026-09-30T08:51:07.194Z


## Write driver status table

In [0]:
(
    driver_stats
        .write # we used just write(batch) because it is not a stream
        .format("delta")
        .mode("overwrite") # it is for update, also delta is parqute whish is immutable so there will be new file and we can to time travle
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(
            "ride_lakehouse_dev.gold.driver_stats"
        )
)

# Second Table Daily stats
This table is for rent company bussines. Each row will represent a day

In [0]:
rides_with_date = (
    silver_rides

    .withColumn(
        "ride_date",
        F.to_date(
            F.col("ride_timestamp")
        )
    )
)

In [0]:
daily_base = (
    rides_with_date.alias("r")
    .join(
        silver_payments.alias("p"),
        F.col("r.ride_id") == F.col("p.ride_id"),
        "left"
    )
)

In [0]:
daily_stats = (
    daily_base

    .groupBy(F.col("r.ride_date"))

    .agg(
        F.countDistinct("r.ride_id").alias("total_rides"),
        F.countDistinct("r.driver_id").alias("unique_drivers"),
        F.countDistinct("r.customer_id").alias("unique_customers"),
        F.round(F.sum("r.distance_km"),2).alias("total_distance_km"),

        F.round(
            F.avg("r.distance_km"),
            2
        ).alias(
            "average_distance_km"
        ),

        F.round(
            F.sum(
                F.coalesce(
                    F.col("p.amount"),
                    F.lit(0.0)
                )
            ),
            2
        ).alias(
            "total_revenue"
        ),

        F.round(
            F.avg("r.fare_amount"),
            2
        ).alias(
            "average_fare"
        )
    )

    .withColumn(
        "_gold_updated_at",
        F.current_timestamp()
    )

    .orderBy(
        F.col("ride_date")
    )
)

In [0]:
(
    daily_stats
        .write

        .format("delta")

        .mode("overwrite")

        .option(
            "overwriteSchema",
            "true"
        )

        .saveAsTable(
            "ride_lakehouse_dev.gold.daily_stats"
        )
)

In [0]:
%sql
-- SELECT *
-- FROM ride_lakehouse_dev.gold.daily_stats
-- ORDER BY ride_date;

ride_date,total_rides,unique_drivers,unique_customers,total_distance_km,average_distance_km,total_revenue,average_fare,_gold_updated_at
2026-09-29,200,40,200,2847.63,14.24,7535.12,37.68,2026-09-30T09:03:52.418Z


In [0]:
%sql
-- SELECT
--     driver_name,
--     total_rides,
--     total_revenue

-- FROM ride_lakehouse_dev.gold.driver_stats

-- ORDER BY total_revenue DESC

-- LIMIT 10;

driver_name,total_rides,total_revenue
Mary Hunt,9,361.46
Harold Williams,8,345.5
Karen Morgan,9,344.99
Joseph Moran,7,327.52
Elizabeth Hernandez,7,315.2
Virginia Schneider,7,283.17
Albert Perez,7,263.68
Vanessa Ray,7,255.34
Joe Wang,5,252.95
Lisa Young,5,251.5
